# Phase 1B — Pre-Extract All Features to `.pt` Files

**ปัญหา:** ทุก epoch ต้อง decode mp3 ทุกไฟล์ใหม่ → GPU idle รอ CPU → training ช้า  
**วิธีแก้:** Extract features **ครั้งเดียว** → save เป็น `.pt` (float16) → load ตรงๆ ทุก epoch → **เร็วขึ้น 10-50×**

## Key design decisions vs Phase 1A
| | Phase 1A (FMADataset) | Phase 1B (PreExtractedFMADataset) |
|---|---|---|
| Per epoch | decode mp3 + extract features | `torch.load()` only |
| Parallelism | ThreadPool (GIL-blocked) | **ProcessPool** (true CPU parallel) |
| Disk format | — | **float16** (~35 KB/file, 50% vs float32) |
| Augmentation | on raw audio (librosa) | **on normalized tensor** (mask=0.0, noise_std∈[0.05,0.15]) |
| Training chunks | 1 random per epoch | **4 chunks** (1 center + 3 random) saved upfront |

## Output structure
```
output/pt_features/
├── training/   → track_{id:06d}_chunk{0-3}.pt  (4 per track)
├── validation/ → track_{id:06d}_chunk0.pt       (center only)
└── test/       → track_{id:06d}_chunk0.pt       (center only)
```
Each file: `{'tabular': float16(88,), 'mel': float16(1,128,130), 'label': int, 'track_id': int}`

---
## Cell 1 — Imports & Configuration

In [1]:
import os
import sys
import json
import pickle
import logging
import random
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import librosa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

# ===========================================================================
# PATHS — relative จาก notebook (อยู่ใน "data prep/")
# ===========================================================================
NOTEBOOK_DIR     = Path().resolve()
OUTPUT_DIR       = NOTEBOOK_DIR / 'output'
METADATA_CSV     = OUTPUT_DIR / 'cleaned_metadata.csv'
SCALER_PKL       = OUTPUT_DIR / 'tabular_scaler.pkl'
SCALING_STATS    = OUTPUT_DIR / 'scaling_stats.json'
CLASS_WEIGHTS_PT = OUTPUT_DIR / 'class_weights.pt'
PT_FEATURES_DIR  = OUTPUT_DIR / 'pt_features'

FMA_AUDIO_DIR    = Path(r'D:\patt\project\pattern-music\FMA_Data\fma_small\fma_small')
FMA_METADATA_CSV = Path(r'D:\patt\project\pattern-music\FMA_Data\fma_metadata\fma_metadata\tracks.csv')

# Import fma_phase1a_data_pipeline จาก folder เดียวกัน
sys.path.insert(0, str(NOTEBOOK_DIR))
from fma_phase1a_data_pipeline import (
    extract_tabular_features,
    extract_mel_spectrogram,
    load_fma_tracks,
    FMADataset,
)

# ===========================================================================
# CONSTANTS
# ===========================================================================
SAMPLE_RATE        = 22050
CHUNK_DURATION     = 3.0
CHUNK_SAMPLES      = int(SAMPLE_RATE * CHUNK_DURATION)  # 66,150
N_MELS             = 128
TARGET_TIME_FRAMES = 130
MEL_SHAPE          = (1, N_MELS, TARGET_TIME_FRAMES)

GENRE_LABELS = [
    'Electronic', 'Experimental', 'Folk', 'Hip-Hop',
    'Instrumental', 'International', 'Pop', 'Rock'
]
NUM_CLASSES = len(GENRE_LABELS)

CORRUPTED_TRACK_IDS = {98565, 98567, 98569, 99134, 108925, 133297, 143992}

N_TRAIN_CHUNKS = 4   # chunk0=center, chunk1-3=random
N_VAL_CHUNKS   = 1   # center only
BATCH_SIZE     = 32
NUM_WORKERS    = 0   # 0 for notebook (Windows: workers cant import notebook-defined classes)

print('✓ Imports and configuration complete')
print(f'  NOTEBOOK_DIR:    {NOTEBOOK_DIR}')
print(f'  PT_FEATURES_DIR: {PT_FEATURES_DIR}')
print(f'  MEL_SHAPE:       {MEL_SHAPE}')
print(f'  Save dtype:      torch.float16  (50% disk vs float32)')

✓ Imports and configuration complete
  NOTEBOOK_DIR:    D:\patt\project\pattern-music\data prep
  PT_FEATURES_DIR: D:\patt\project\pattern-music\data prep\output\pt_features
  MEL_SHAPE:       (1, 128, 130)
  Save dtype:      torch.float16  (50% disk vs float32)


---
## Cell 2 — Load Phase 1A Artifacts

In [2]:
# --- StandardScaler สำหรับ tabular (88 dims) ---
with open(SCALER_PKL, 'rb') as f:
    tabular_scaler = pickle.load(f)
print(f'✓ tabular_scaler.pkl  ({type(tabular_scaler).__name__})')

# --- Global mel statistics ---
with open(SCALING_STATS, 'r') as f:
    scaling_stats = json.load(f)
MEL_MEAN = float(scaling_stats['mel_mean'])
MEL_STD  = float(scaling_stats['mel_std'])
print(f'✓ scaling_stats.json  mel_mean={MEL_MEAN:.4f} dB, mel_std={MEL_STD:.4f} dB')

# --- Class weights ---
class_weights = torch.load(CLASS_WEIGHTS_PT, weights_only=True)
print(f'✓ class_weights.pt    shape={tuple(class_weights.shape)}')
for i, (g, w) in enumerate(zip(GENRE_LABELS, class_weights)):
    print(f'    [{i}] {g:15s} → {w:.4f}')

✓ tabular_scaler.pkl  (StandardScaler)
✓ scaling_stats.json  mel_mean=-41.7546 dB, mel_std=13.7384 dB
✓ class_weights.pt    shape=(8,)
    [0] Electronic      → 0.9992
    [1] Experimental    → 0.9992
    [2] Folk            → 0.9992
    [3] Hip-Hop         → 1.0017
    [4] Instrumental    → 0.9992
    [5] International   → 1.0042
    [6] Pop             → 0.9980
    [7] Rock            → 0.9992


---
## Cell 3 — Load Metadata

In [3]:
# ลอง cleaned_metadata.csv ก่อน → fallback ไป tracks.csv
if METADATA_CSV.exists() and {'track_id','genre_idx','split'}.issubset(
        pd.read_csv(METADATA_CSV, nrows=0).columns):
    metadata_df = pd.read_csv(METADATA_CSV)
    print(f'✓ Loaded cleaned_metadata.csv ({len(metadata_df):,} tracks)')
else:
    print('⚠ Falling back to load_fma_tracks()')
    metadata_df = load_fma_tracks(FMA_METADATA_CSV)

# กรอง corrupted tracks
before = len(metadata_df)
metadata_df = metadata_df[~metadata_df['track_id'].isin(CORRUPTED_TRACK_IDS)].copy()
print(f'  Removed {before - len(metadata_df)} corrupted → {len(metadata_df):,} tracks remain')

split_counts = metadata_df['split'].value_counts()
print('\nSplit distribution:')
for split, count in split_counts.items():
    print(f'  {split:12s}: {count:,}')

metadata_df.head(3)

✓ Loaded cleaned_metadata.csv (7,985 tracks)
  Removed 0 corrupted → 7,985 tracks remain

Split distribution:
  training    : 6,387
  test        : 800
  validation  : 798


,track_id,genre_top,genre_idx,split,duration_actual,rms_mean,file_size_kb
0,2,Hip-Hop,3,training,29.976599,0.145215,938.220703
1,5,Hip-Hop,3,training,30.002721,0.148774,939.042969
2,10,Pop,6,training,29.976599,0.188172,704.140625


---
## Cell 4 — Disk Space Estimation

คำนวณ disk space ก่อน extract — บันทึกเป็น **float16** (ลดเหลือ ~50% เทียบกับ float32)

In [4]:
# float16: 2 bytes ต่อ element (แทน float32: 4 bytes)
tabular_bytes = 88 * 2                  # 176 bytes
mel_bytes     = 1 * 128 * 130 * 2      # 33,280 bytes
overhead      = 2000                    # torch serialization + dict overhead
bytes_per_file = tabular_bytes + mel_bytes + overhead  # ≈ 35,456 bytes ≈ 35 KB

n_train = split_counts.get('training', 0)
n_val   = split_counts.get('validation', 0)
n_test  = split_counts.get('test', 0)

files_train = n_train * N_TRAIN_CHUNKS
files_val   = n_val   * N_VAL_CHUNKS
files_test  = n_test  * N_VAL_CHUNKS
total_files = files_train + files_val + files_test

total_bytes = total_files * bytes_per_file
total_gb    = total_bytes / (1024 ** 3)

# เปรียบเทียบกับ float32
float32_gb = total_gb * 2

print('=' * 62)
print('  DISK SPACE ESTIMATION  (float16 — 2 bytes/element)')
print('=' * 62)
print(f'  Per .pt file:         ~{bytes_per_file/1024:.0f} KB')
print(f'    tabular 88×f16:      {tabular_bytes} B')
print(f'    mel 1×128×130×f16:   {mel_bytes:,} B')
print(f'    torch overhead:      ~{overhead} B')
print()
print(f'  training  : {n_train:,} × {N_TRAIN_CHUNKS} chunks = {files_train:,} files')
print(f'  validation: {n_val:,} × 1 chunk  = {files_val:,} files')
print(f'  test      : {n_test:,} × 1 chunk  = {files_test:,} files')
print(f'  ──────────────────────────────────────────────────')
print(f'  Total files:          {total_files:,}')
print(f'  Estimated (float16):  {total_gb:.2f} GB')
print(f'  Would be (float32):   {float32_gb:.2f} GB  → float16 saves {float32_gb-total_gb:.2f} GB')
print('=' * 62)
print(f'\n⚠  Proceeding will use ~{total_gb:.2f} GB of disk space.')
print('   Run Cell 5 to start extraction.')
print('   (Skip to Cell 8 if .pt files already exist)')

  DISK SPACE ESTIMATION  (float16 — 2 bytes/element)
  Per .pt file:         ~35 KB
    tabular 88×f16:      176 B
    mel 1×128×130×f16:   33,280 B
    torch overhead:      ~2000 B

  training  : 6,387 × 4 chunks = 25,548 files
  validation: 798 × 1 chunk  = 798 files
  test      : 800 × 1 chunk  = 800 files
  ──────────────────────────────────────────────────
  Total files:          27,146
  Estimated (float16):  0.90 GB
  Would be (float32):   1.79 GB  → float16 saves 0.90 GB

⚠  Proceeding will use ~0.90 GB of disk space.
   Run Cell 5 to start extraction.
   (Skip to Cell 8 if .pt files already exist)


---
## Cell 5 — Core Extraction Functions

> **ℹ️ Windows + ProcessPoolExecutor:** worker function (`extract_and_save_track`) ต้อง import จาก module  
> ไม่ define ใน cell เอง — ทำให้ function picklable สำหรับ child processes

In [5]:
# ===========================================================================
# Import worker function จาก .py module (picklable บน Windows)
# ⚠️ ถ้า define ใน cell จะ NOT picklable → ProcessPoolExecutor จะ crash
# ===========================================================================
from fma_phase1b_feature_extraction import extract_and_save_track

# Helper functions สำหรับ extraction (define ใน cell ได้เพราะไม่ต้อง pickle)
def extract_chunk(
    audio: np.ndarray,
    chunk_samples: int,
    mode: str = 'center',
    rng: Optional[np.random.RandomState] = None,
) -> np.ndarray:
    """
    ตัด chunk ขนาด chunk_samples จาก full audio
    mode='center' → center crop (deterministic)
    mode='random' → random start (training augmentation)
    """
    total = len(audio)
    if total < chunk_samples:
        padded = np.zeros(chunk_samples, dtype=np.float32)
        padded[:total] = audio
        return padded
    if mode == 'center':
        start = (total - chunk_samples) // 2
    else:
        if rng is None:
            rng = np.random.RandomState()
        start = rng.randint(0, total - chunk_samples + 1)
    return audio[start:start + chunk_samples].astype(np.float32)


print('✓ Worker function imported from fma_phase1b_feature_extraction.py')
print('  (importable module → picklable → safe for ProcessPoolExecutor on Windows)')
print(f'  extract_and_save_track: {extract_and_save_track}')

✓ Worker function imported from fma_phase1b_feature_extraction.py
  (importable module → picklable → safe for ProcessPoolExecutor on Windows)
  extract_and_save_track: <function extract_and_save_track at 0x000002D24BAB2700>


---
## Cell 6 — Run Pre-Extraction

> **⚠ ใช้เวลานาน** (~20-40 นาที สำหรับ 8000 tracks)  
> `ProcessPoolExecutor(max_workers=4)` — true CPU parallelism (librosa FFT เป็น CPU-bound)  
> Resume-safe: `overwrite=False` จะ skip tracks ที่ extract ไปแล้ว

In [6]:
MAX_WORKERS = 4      # ProcessPoolExecutor workers — ปรับตาม CPU cores
OVERWRITE   = False  # True = overwrite .pt ที่มีอยู่แล้ว

# สร้าง output directories
for s in ['training', 'validation', 'test']:
    (PT_FEATURES_DIR / s).mkdir(parents=True, exist_ok=True)
print(f'✓ Output dirs: {PT_FEATURES_DIR}')

# ===========================================================================
# EXTRACTION LOOP
# ⚠️ Windows: ProcessPoolExecutor ต้องอยู่ภายใต้ if __name__ == '__main__':
#    ใน notebook ทำไม่ได้ตรงๆ — แต่ใช้ได้เพราะ worker function import จาก module
#    (ไม่ได้ defined ใน __main__ ของ notebook)
# ===========================================================================
extraction_results = {}
total_start = time.perf_counter()

for split_name in ['training', 'validation', 'test']:
    split_df   = metadata_df[metadata_df['split'] == split_name].reset_index(drop=True)
    out_dir    = PT_FEATURES_DIR / split_name
    n_chunks   = N_TRAIN_CHUNKS if split_name == 'training' else N_VAL_CHUNKS
    results    = {'success': 0, 'failed': 0, 'errors': []}

    print(f'\n[{split_name.upper()}] {len(split_df):,} tracks → {len(split_df)*n_chunks:,} files')

    # หา tracks ที่ยังไม่ได้ extract
    rows_to_process = []
    skipped = 0
    for _, row in split_df.iterrows():
        tid = int(row['track_id'])
        if tid in CORRUPTED_TRACK_IDS:
            skipped += 1
            continue
        chunk0 = out_dir / f'track_{tid:06d}_chunk0.pt'
        if not OVERWRITE and chunk0.exists():
            results['success'] += n_chunks
            skipped += 1
            continue
        rows_to_process.append(row)

    if skipped > 0:
        print(f'  Skipping {skipped} tracks (already done or corrupted)')
    if not rows_to_process:
        print(f'  ✓ All tracks already extracted')
        extraction_results[split_name] = results
        continue

    print(f'  Processing {len(rows_to_process):,} tracks (ProcessPoolExecutor, {MAX_WORKERS} workers)...')

    # ProcessPoolExecutor — true CPU parallelism สำหรับ librosa CPU-bound operations
    futures = {}
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for row in rows_to_process:
            future = executor.submit(
                extract_and_save_track,  # imported from module → picklable
                int(row['track_id']),
                FMA_AUDIO_DIR,
                int(row['genre_idx']),
                split_name,
                out_dir,
                tabular_scaler,
                MEL_MEAN,
                MEL_STD,
            )
            futures[future] = int(row['track_id'])

        pbar = tqdm(as_completed(futures), total=len(futures),
                    desc=f'  {split_name[:5]}', unit='track')
        for future in pbar:
            track_id, saved_paths, error = future.result()
            if error is None:
                results['success'] += len(saved_paths)
            else:
                results['failed'] += 1
                results['errors'].append(f'track_{track_id:06d}: {error}')

    extraction_results[split_name] = results
    print(f'  ✓ Success: {results["success"]:,} files')
    if results['failed']:
        print(f'  ✗ Failed:  {results["failed"]:,} tracks')
        for e in results['errors'][:5]:
            print(f'      {e}')

# Final summary
elapsed = time.perf_counter() - total_start
total_success = sum(r['success'] for r in extraction_results.values())
total_failed  = sum(r['failed']  for r in extraction_results.values())
actual_bytes  = sum(
    f.stat().st_size
    for s in ['training','validation','test']
    for f in (PT_FEATURES_DIR/s).glob('*.pt')
)
print(f'\n{"="*62}')
print(f'  EXTRACTION COMPLETE')
print(f'  Total .pt files: {total_success:,}   Failed: {total_failed:,}')
print(f'  Time elapsed:    {elapsed/60:.1f} min')
print(f'  Actual disk:     {actual_bytes/(1024**3):.3f} GB (float16)')
print(f'{"="*62}')

✓ Output dirs: D:\patt\project\pattern-music\data prep\output\pt_features

[TRAINING] 6,387 tracks → 25,548 files
  Skipping 6387 tracks (already done or corrupted)
  ✓ All tracks already extracted

[VALIDATION] 798 tracks → 798 files
  Skipping 798 tracks (already done or corrupted)
  ✓ All tracks already extracted

[TEST] 800 tracks → 800 files
  Skipping 800 tracks (already done or corrupted)
  ✓ All tracks already extracted

  EXTRACTION COMPLETE
  Total .pt files: 27,146   Failed: 0
  Time elapsed:    0.0 min
  Actual disk:     0.884 GB (float16)


---
## Cell 7 — Tensor-Based Augmentation

Augment บน **normalized tensor** (mean≈0, std≈1) — ไม่ใช้ librosa เลย

In [7]:
def spec_augment(
    mel: torch.Tensor,
    freq_mask_param: int = 15,
    time_mask_param: int = 20,
    n_freq_masks: int = 2,
    n_time_masks: int = 2,
) -> torch.Tensor:
    """
    SpecAugment บน normalized mel tensor (1, 128, 130)

    fill_val = 0.0 เพราะ data normalized แล้ว (mean≈0)
    → 0.0 คือ mean ของ distribution → masked regions ไม่ distort stats
    """
    mel = mel.clone()
    # 0.0 = mean ของ normalized data — ไม่ต้องคำนวณ mean สดๆ
    fill_val = 0.0
    _, n_mels, n_frames = mel.shape

    for _ in range(n_freq_masks):
        f  = random.randint(0, freq_mask_param)
        f0 = random.randint(0, max(0, n_mels - f))
        mel[:, f0:f0+f, :] = fill_val

    for _ in range(n_time_masks):
        t  = random.randint(0, time_mask_param)
        t0 = random.randint(0, max(0, n_frames - t))
        mel[:, :, t0:t0+t] = fill_val

    return mel


def add_gaussian_noise(
    mel: torch.Tensor,
    noise_std: Optional[float] = None,
) -> torch.Tensor:
    """
    Gaussian noise บน normalized mel tensor (1, 128, 130)

    ใช้ noise_std ∈ [0.05, 0.15] — เหมาะกับ normalized data (std≈1)
    ไม่จำเป็นต้องคำนวณ SNR เพราะ signal scale รู้อยู่แล้ว (~1.0)
    """
    if noise_std is None:
        noise_std = random.uniform(0.05, 0.15)
    return mel + torch.randn_like(mel) * noise_std


# Smoke test
dummy = torch.randn(1, 128, 130)
aug   = spec_augment(dummy)
noisy = add_gaussian_noise(dummy, noise_std=0.1)

print('✓ Augmentation functions defined (tensor-based, no librosa)')
print(f'  spec_augment:      fill=0.0 (normalized mean), shape={tuple(aug.shape)}')
print(f'  add_gaussian_noise: noise_std~U(0.05,0.15),   shape={tuple(noisy.shape)}')
print(f'  Noise added:       mean diff = {(noisy-dummy).mean():.4f}, std = {(noisy-dummy).std():.4f}')

✓ Augmentation functions defined (tensor-based, no librosa)
  spec_augment:      fill=0.0 (normalized mean), shape=(1, 128, 130)
  add_gaussian_noise: noise_std~U(0.05,0.15),   shape=(1, 128, 130)
  Noise added:       mean diff = 0.0006, std = 0.1002


---
## Cell 8 — `PreExtractedFMADataset` Class

In [8]:
class PreExtractedFMADataset(Dataset):
    """
    Fast PyTorch Dataset — load .pt files แทน mp3

    __getitem__:
      1. torch.load()          — no mp3 decode
      2. .to(torch.float32)    — convert float16 → float32
      3. augmentation          — training only, บน normalized tensor
      4. return (tabular, mel, label)
    """

    def __init__(self, pt_dir: Path, augment: bool = False):
        self.pt_dir  = Path(pt_dir)
        self.augment = augment
        self.files   = sorted(self.pt_dir.glob('*.pt'))

        if not self.files:
            raise FileNotFoundError(
                f'No .pt files in {self.pt_dir} — run Cell 6 first'
            )
        print(f'  [{self.pt_dir.name}] {len(self.files):,} files, augment={augment}')

    def __len__(self):
        return len(self.files)

    def __getitem__(
        self, idx: int
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
            tabular: FloatTensor (88,)       — float16→float32
            mel:     FloatTensor (1,128,130) — float16→float32, optionally augmented
            label:   LongTensor  scalar
        """
        data    = torch.load(self.files[idx], weights_only=True)
        # float16 → float32 ก่อนส่งเข้า model (float16 ไม่ support ทุก operation)
        tabular = data['tabular'].to(torch.float32)   # (88,)
        mel     = data['mel'].to(torch.float32)       # (1, 128, 130)
        label   = torch.tensor(data['label'], dtype=torch.long)

        # Augmentation — training only (val/test: ไม่ augment เด็ดขาด)
        if self.augment:
            # SpecAugment: mask = 0.0 (= normalized mean) — p=0.5
            if random.random() < 0.5:
                mel = spec_augment(mel)
            # Gaussian noise: noise_std ~ U(0.05, 0.15) — p=0.3
            if random.random() < 0.3:
                mel = add_gaussian_noise(mel)

        return tabular, mel, label


# Test instantiation + shapes
print('✓ PreExtractedFMADataset class defined')
print('Testing instantiation...')

for split_name in ['training', 'validation', 'test']:
    split_dir = PT_FEATURES_DIR / split_name
    if split_dir.exists() and any(split_dir.glob('*.pt')):
        ds = PreExtractedFMADataset(split_dir, augment=(split_name=='training'))
        tab, mel, lbl = ds[0]
        assert tab.shape == (88,),         f'tabular: {tab.shape}'
        assert mel.shape == (1,128,130),   f'mel: {mel.shape}'
        assert tab.dtype == torch.float32, f'tabular dtype: {tab.dtype}'
        assert mel.dtype == torch.float32, f'mel dtype: {mel.dtype}'
        print(f'    ✓ [{split_name}] shapes OK, dtypes float32 (loaded from float16)')
    else:
        print(f'    [{split_name}] No .pt files yet — run Cell 6 first')

✓ PreExtractedFMADataset class defined
Testing instantiation...
  [training] 25,548 files, augment=True
    ✓ [training] shapes OK, dtypes float32 (loaded from float16)
  [validation] 798 files, augment=False
    ✓ [validation] shapes OK, dtypes float32 (loaded from float16)
  [test] 800 files, augment=False
    ✓ [test] shapes OK, dtypes float32 (loaded from float16)


---
## Cell 9 — `create_fast_dataloaders()` Factory

> **⚠️ Windows Jupyter:** `persistent_workers=False` — ป้องกัน zombie processes และ RAM leak  
> `persistent_workers=True` ได้เมื่อรันเป็น `.py` script

In [9]:
def create_fast_dataloaders(
    pt_base_dir: Path = PT_FEATURES_DIR,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
    use_weighted_sampler: bool = False,   # False: ไม่อ่าน disk → instant (default)
    class_weights_tensor: Optional[torch.Tensor] = None,
    metadata_df=None,                     # required เมื่อ use_weighted_sampler=True
    persistent_workers: bool = False,     # False สำหรับ notebook (Windows compat)
) -> Dict[str, DataLoader]:
    """
    สร้าง DataLoaders จาก pre-extracted .pt files

    use_weighted_sampler=False (default): shuffle แบบปกติ — instant
    use_weighted_sampler=True:  ใช้ filename-based label lookup (O(1)) —
        ต้องส่ง metadata_df (DataFrame ที่มีคอลัมน์ track_id, genre_idx)

    persistent_workers:
        False → notebook (Windows Jupyter zombie process fix)
        True  → .py script (ลด worker spawn overhead ระหว่าง batches)
    """
    pt_base_dir = Path(pt_base_dir)

    if class_weights_tensor is None and CLASS_WEIGHTS_PT.exists():
        class_weights_tensor = torch.load(CLASS_WEIGHTS_PT, weights_only=True)

    loaders = {}

    for split_name in ['training', 'validation', 'test']:
        split_dir = pt_base_dir / split_name
        is_train  = (split_name == 'training')

        dataset = PreExtractedFMADataset(split_dir, augment=is_train)

        # WeightedRandomSampler — optional (dataset balanced อยู่แล้ว)
        sampler = None
        if is_train and use_weighted_sampler and class_weights_tensor is not None:
            if metadata_df is None:
                raise ValueError(
                    'metadata_df is required when use_weighted_sampler=True. '
                    'Pass metadata_df=metadata_df to avoid O(N) disk reads.'
                )
            # Fast path: parse track_id from filename (zero disk reads)
            # Filename pattern: track_000002_chunk0.pt -> track_id = 2
            tid_to_label = dict(
                zip(
                    metadata_df['track_id'].astype(int),
                    metadata_df['genre_idx'].astype(int),
                )
            )
            sample_weights = [
                class_weights_tensor[
                    tid_to_label.get(int(f.stem.split('_')[1]), 0)
                ].item()
                for f in dataset.files   # O(1) dict lookup, zero disk reads
            ]
            sampler = WeightedRandomSampler(
                weights     = torch.tensor(sample_weights, dtype=torch.double),
                num_samples = len(dataset),
                replacement = True,
            )

        loader = DataLoader(
            dataset,
            batch_size         = batch_size,
            sampler            = sampler,
            shuffle            = (is_train and sampler is None),
            num_workers        = num_workers,
            pin_memory         = True,
            drop_last          = is_train,
            # Windows Jupyter: False ป้องกัน zombie processes / RAM leak
            persistent_workers = persistent_workers,
        )

        loaders[split_name] = loader
        print(
            f'  [{split_name}] {len(dataset):,} samples, '
            f'{len(loader):,} batches, augment={is_train}, '
            f'persistent_workers={persistent_workers}'
        )

    return loaders


print('create_fast_dataloaders() defined')
print('Creating DataLoaders (use_weighted_sampler=False for instant startup)...')
fast_loaders = create_fast_dataloaders(
    pt_base_dir          = PT_FEATURES_DIR,
    batch_size           = BATCH_SIZE,
    num_workers          = NUM_WORKERS,
    use_weighted_sampler = False,    # instant: no disk reads
    # To enable: use_weighted_sampler=True, metadata_df=metadata_df
    class_weights_tensor = class_weights,
    metadata_df          = metadata_df,
    persistent_workers   = False,    # notebook: False (Windows zombie fix)
)
print(f'Ready: {list(fast_loaders.keys())}')


create_fast_dataloaders() defined
Creating DataLoaders (use_weighted_sampler=False for instant startup)...
  [training] 25,548 files, augment=True
  [training] 25,548 samples, 798 batches, augment=True, persistent_workers=False
  [validation] 798 files, augment=False
  [validation] 798 samples, 25 batches, augment=False, persistent_workers=False
  [test] 800 files, augment=False
  [test] 800 samples, 25 batches, augment=False, persistent_workers=False
Ready: ['training', 'validation', 'test']


---
## Cell 10 — Verification: Shapes + NaN/Inf

In [10]:
print('=' * 62)
print('  VERIFICATION: Shape + NaN/Inf Checks')
print('=' * 62)

for split_name in ['training', 'validation']:
    loader = fast_loaders[split_name]
    tabular_b, mel_b, label_b = next(iter(loader))
    bs = tabular_b.shape[0]

    print(f'\n[{split_name.upper()}]')
    print(f'  tabular: {tuple(tabular_b.shape)}   dtype={tabular_b.dtype}')
    print(f'  mel:     {tuple(mel_b.shape)}  dtype={mel_b.dtype}')
    print(f'  label:   {tuple(label_b.shape)}            dtype={label_b.dtype}')

    # Shape assertions
    assert tabular_b.shape == (bs, 88),         f'tabular: {tabular_b.shape}'
    assert mel_b.shape     == (bs, 1, 128, 130), f'mel: {mel_b.shape}'
    assert label_b.shape   == (bs,),             f'label: {label_b.shape}'

    # dtype must be float32 (converted from float16 in __getitem__)
    assert tabular_b.dtype == torch.float32, f'tabular dtype: {tabular_b.dtype}'
    assert mel_b.dtype     == torch.float32, f'mel dtype: {mel_b.dtype}'

    # NaN/Inf
    assert not torch.isnan(tabular_b).any(), 'NaN in tabular!'
    assert not torch.isnan(mel_b).any(),     'NaN in mel!'
    assert not torch.isinf(tabular_b).any(), 'Inf in tabular!'
    assert not torch.isinf(mel_b).any(),     'Inf in mel!'

    print(f'  ✓ shapes OK | dtype=float32 (from float16) | no NaN/Inf')
    print(f'  mel stats:     min={mel_b.min():.3f}, max={mel_b.max():.3f}, mean={mel_b.mean():.3f}')
    print(f'  tabular stats: min={tabular_b.min():.3f}, max={tabular_b.max():.3f}, mean={tabular_b.mean():.3f}')
    print(f'  label range:   {label_b.min().item()} – {label_b.max().item()}')

print('\n✓ All assertions PASSED')

  VERIFICATION: Shape + NaN/Inf Checks

[TRAINING]
  tabular: (32, 88)   dtype=torch.float32
  mel:     (32, 1, 128, 130)  dtype=torch.float32
  label:   (32,)            dtype=torch.int64
  ✓ shapes OK | dtype=float32 (from float16) | no NaN/Inf
  mel stats:     min=-3.295, max=3.309, mean=-0.038
  tabular stats: min=-4.637, max=5.703, mean=-0.058
  label range:   0 – 7

[VALIDATION]
  tabular: (32, 88)   dtype=torch.float32
  mel:     (32, 1, 128, 130)  dtype=torch.float32
  label:   (32,)            dtype=torch.int64
  ✓ shapes OK | dtype=float32 (from float16) | no NaN/Inf
  mel stats:     min=-2.783, max=3.039, mean=0.063
  tabular stats: min=-4.117, max=5.648, mean=-0.142
  label range:   0 – 7

✓ All assertions PASSED


---
## Cell 11 — Speedup Benchmark: `.pt` vs `mp3`

In [11]:
print('=' * 62)
print('  SPEEDUP BENCHMARK: .pt (float16) vs mp3 loading')
print('=' * 62)

N_BENCH = 5  # batches

# --- .pt loading ---
t0 = time.perf_counter()
for i, _ in enumerate(fast_loaders['training']):
    if i >= N_BENCH - 1:
        break
pt_time = time.perf_counter() - t0
print(f'  .pt  ({N_BENCH} batches): {pt_time:.3f}s  ({pt_time/N_BENCH:.3f}s/batch)')

# --- mp3 loading (FMADataset เดิม, num_workers=0) ---
print(f'  Benchmarking mp3 loading...')
try:
    mp3_ds     = FMADataset(metadata_df, FMA_AUDIO_DIR, split='training')
    mp3_loader = DataLoader(mp3_ds, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=0, pin_memory=False)
    t0 = time.perf_counter()
    for i, _ in enumerate(mp3_loader):
        if i >= N_BENCH - 1:
            break
    mp3_time = time.perf_counter() - t0

    speedup = mp3_time / pt_time if pt_time > 0 else float('inf')
    target  = '✓ TARGET MET (≥10×)!' if speedup >= 10 else '⚠ below 10× target'

    print(f'  mp3  ({N_BENCH} batches): {mp3_time:.3f}s  ({mp3_time/N_BENCH:.3f}s/batch)')
    print(f'  {"─"*40}')
    print(f'  Speedup: {speedup:.1f}×  {target}')
    print(f'  {"─"*40}')

except Exception as e:
    print(f'  mp3 benchmark skipped: {e}')

  SPEEDUP BENCHMARK: .pt (float16) vs mp3 loading
  .pt  (5 batches): 1.299s  (0.260s/batch)
  Benchmarking mp3 loading...


19:11:28 | INFO | [training] Dataset ready: 6387 tracks, 8 genres


  mp3  (5 batches): 18.051s  (3.610s/batch)
  ────────────────────────────────────────
  Speedup: 13.9×  ✓ TARGET MET (≥10×)!
  ────────────────────────────────────────


---
## Cell 12 — Plot Sample Spectrograms

In [12]:
tabular_b, mel_b, label_b = next(iter(fast_loaders['training']))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for i, ax in enumerate(axes.flatten()):
    spec  = mel_b[i, 0].numpy()   # (128, 130) — normalized, float32
    genre = GENRE_LABELS[label_b[i].item()]
    im = ax.imshow(
        spec, aspect='auto', origin='lower', cmap='magma',
        extent=[0, CHUNK_DURATION, 0, SAMPLE_RATE // 2],
    )
    ax.set_title(f'{genre}  |  min={spec.min():.2f} max={spec.max():.2f}', fontsize=10)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Freq (Hz)')
    fig.colorbar(im, ax=ax, format='%.1f')

fig.suptitle(
    'Phase 1B — Mel Spectrograms (globally normalized, float16→float32, training augmented)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()

save_path = OUTPUT_DIR / 'phase1b_sample_spectrograms.png'
fig.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {save_path.name}')

✓ Saved: phase1b_sample_spectrograms.png


---
## Cell 13 — Summary & Import Snippet for Phase 2

In [13]:
file_counts = {
    s: len(list((PT_FEATURES_DIR/s).glob('*.pt')))
    for s in ['training','validation','test']
}
actual_bytes = sum(
    f.stat().st_size
    for s in file_counts for f in (PT_FEATURES_DIR/s).glob('*.pt')
)

print('=' * 62)
print('  PHASE 1B COMPLETE')
print('=' * 62)
print(f'  .pt files:')
for s, c in file_counts.items():
    print(f'    {s:12s}: {c:,}')
print(f'  Total:          {sum(file_counts.values()):,} files')
print(f'  Disk (float16): {actual_bytes/(1024**3):.3f} GB')
print()

n_train_orig = len(metadata_df[metadata_df["split"]=="training"])
print(f'  Training expansion:')
print(f'    Original tracks: {n_train_orig:,}')
print(f'    After 4× chunk:  {file_counts["training"]:,} .pt files')
print(f'    Expansion ratio: {file_counts["training"]/n_train_orig:.1f}×')
print()
print('=' * 62)
print('  USE IN PHASE 2 (import from .py module):')
print('=' * 62)
print('''
from fma_phase1b_feature_extraction import (
    PreExtractedFMADataset,
    create_fast_dataloaders,
    PT_FEATURES_DIR,
    spec_augment,
    add_gaussian_noise,
)

loaders = create_fast_dataloaders(
    pt_base_dir        = PT_FEATURES_DIR,
    persistent_workers = True,   # True for .py script
)
# loaders["training"]   → training loop
# loaders["validation"] → early stopping / hyperparam tuning
# loaders["test"]       → final evaluation (touch once!)
''')

  PHASE 1B COMPLETE
  .pt files:
    training    : 25,548
    validation  : 798
    test        : 800
  Total:          27,146 files
  Disk (float16): 0.884 GB

  Training expansion:
    Original tracks: 6,387
    After 4× chunk:  25,548 .pt files
    Expansion ratio: 4.0×

  USE IN PHASE 2 (import from .py module):

from fma_phase1b_feature_extraction import (
    PreExtractedFMADataset,
    create_fast_dataloaders,
    PT_FEATURES_DIR,
    spec_augment,
    add_gaussian_noise,
)

loaders = create_fast_dataloaders(
    pt_base_dir        = PT_FEATURES_DIR,
    persistent_workers = True,   # True for .py script
)
# loaders["training"]   → training loop
# loaders["validation"] → early stopping / hyperparam tuning
# loaders["test"]       → final evaluation (touch once!)

